Context:  
This notebook was solely used during the earliest phases of the project. It experiments with training a MedRoBERTa model to directly predict the score, without considering the components separatly. The focus of the project shifted quickly towards the identification of the components. 

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report
import pandas as pd
import re
import pprint
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import numpy as np

from snorkel.labeling.analysis import LFAnalysis
from snorkel.labeling import labeling_function, PandasLFApplier
from snorkel.labeling.model import LabelModel
import import_ipynb
from snorkel_component_based import (
    lf_structured_chf,
    lf_chf_positive,
    lf_chf_negation,
    lf_chf_tabular_format,
)

In [ ]:
# -----------------------------
# Step 1: Load model
# -----------------------------
model_path = r"..\code\Chadsvasc score extraction\Expert-based\models\medRoberta_expert_component_2"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()
print("Step 1 completed: model loaded")

In [ ]:
# -----------------------------
# Step 2: Load test dataset and split the text in sections
# -----------------------------

test_df = pd.read_feather(
    r"..\code\Chadsvasc score extraction\datasets\largest_labeled_chads_set.feather"
)

print(test_df.columns)

section_titles = [
    "allergie",
    "anamnese",
    "beleid",
    "besluit",
    "cardiale risicofactoren",
    "ecg",
    "echocardiografie",
    "elektrolyten",
    "klinisch onderzoek",
    "laboratoriumresultaten",
    "medische antecedenten",
    "parameters",
    "planning",
    "rx thorax",
    "telemetrie",
    "thuismedicatie",
]

section_pattern = re.compile(r"^([A-Z\s]+?)\s*(?:[:\n]|$)", re.MULTILINE)


def split_sections(text):
    sections = {}
    matches = list(section_pattern.finditer(text))
    for i, match in enumerate(matches):
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        section_name = match.group(1).upper()
        sections[section_name] = text[start:end].strip()
    return sections


test_df["sections"] = test_df["text"].apply(split_sections)

relevant_sections = [
    "BESLUIT",
    "BESPREKING EN BESLUIT",
    "MEDISCHE ANTECEDENTEN",
    "CARDIALE RISICOFACTOREN",
    "BESPREKING",
]
test_df["medRoberta_text"] = test_df["sections"].apply(
    lambda x: " ".join(str(x.get(sec, "")).lower() for sec in relevant_sections)
)

print("Column 'sections' successfully added")

In [ ]:
texts = test_df["text"].tolist()
inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt",
    max_length=512,
)

In [ ]:
components = ["C", "H", "A2", "D", "S", "V", "A", "Sc"]

# -----------------------------
# Step 3: Evaluate on new dataset and create column 'chadsvasc_expert_component'
# -----------------------------
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).int().numpy()

In [ ]:
for i, name in enumerate(components):
    test_df[name] = preds[:, i]
test_df["chadsvasc_expert_component"] = preds.sum(axis=1)
print(test_df.head())
print("Step 3 completed: predictions added to dataframe")

In [ ]:
filtered_df = test_df[~test_df["label"].isin([10, 13])]

filtered_df.to_feather(
    r"..\code\Chadsvasc score extraction\Expert-based\feather_files\evaluation_with_chadsvasc_expert_component.feather"
)

# -----------------------------
# Step 4: Compare column 'chadsvasc_expert_component' with 'label' column
# -----------------------------
print(
    classification_report(
        filtered_df["label"], filtered_df["chadsvasc_expert_component"], digits=3
    )
)

These results look very bad, but it is treating the problem as an exact class equality. We look for more relevant metrics to evaluate the performance, taking the ordinal nature of the task into account. More relevant metrics seem to be:
- MAE: Mean Absolute Error
- RMSE: Root Mean Squared Error: Quadratic mean of the differences between the observed values and the predicted ones. 
- Within 1 and Within 2 distances


In [ ]:
filtered_df["score_diff"] = (
    filtered_df["chadsvasc_expert_component"] - filtered_df["label"]
)

print(filtered_df[["chadsvasc_expert_component", "label", "score_diff"]].describe())

print("Exact match rate:", (filtered_df["score_diff"] == 0).mean())
print("Difference counts:")
print(filtered_df["score_diff"].value_counts())

true = filtered_df["label"]
pred = filtered_df["chadsvasc_expert_component"]

mae = mean_absolute_error(true, pred)
rmse = root_mean_squared_error(true, pred)

within_1 = np.mean(np.abs(true - pred) <= 1)
within_2 = np.mean(np.abs(true - pred) <= 2)

print("MAE:", mae)
print("RMSE:", rmse)
print("Within ±1 accuracy:", within_1)
print("Within ±2 accuracy:", within_2)

# Inspection of the outliers

In [ ]:
import re
import textwrap

window = 300
line_width = 100

chads_pattern = r"""
    \b(?:CHA[²2?-]+?DS?[2²?-]+?|CHADVASC?|CHADS-?2?)
    [\s\-\_]*                            
    (?:VAS?C?2?)?                           
"""

# Filter extreme outliers
threshold = 4
outliers = filtered_df[filtered_df["score_diff"].abs() >= threshold]

print(f"Found {len(outliers)} extreme outliers (|score_diff| >= {threshold})")

for i, row in enumerate(outliers.itertuples(), start=1):
    text = row.text
    label = row.label
    predicted = row.chadsvasc_expert_component  # or row.chadsvasc_score
    diff = row.score_diff

    print("\n" + "=" * 30)
    print(f"Outlier {i}")
    print(f"True label: {label}")
    print(f"Predicted: {predicted}")
    print(f"Score diff: {diff}")

    component_values = [getattr(row, col) for col in components]
    col_width = 5
    header = (
        "".join(f"{comp:>{col_width}}" for comp in components)
        + f"{'Total':>{col_width}}"
    )
    print(header)
    row = (
        "".join(f"{val:>{col_width}}" for val in component_values)
        + f"{predicted:>{col_width}}"
    )
    print(row)

    print("=" * 30 + "\n")

    # Search for CHADS-VASc mention
    match = re.search(chads_pattern, text, flags=re.IGNORECASE | re.VERBOSE)

    if match:
        start = max(match.start() - window, 0)
        end = min(match.end() + window, len(text))
        context = text[start:end]

        wrapped_context = "\n".join(textwrap.wrap(context, width=line_width))

        print("Context around CHADS-VASc:")
        print(wrapped_context)
        print(f"\nMatched phrase: {match.group(0)}")

    else:
        print("⚠ No CHADS-VASc pattern found in the text.")
        wrapped_text = "\n".join(textwrap.wrap(text, width=line_width))
        print("Full note text (wrapped):\n")
        print(wrapped_text)

# Check coverages of labeling functions

In [ ]:
components = ["C", "H", "A2", "D", "S", "V", "A", "Sc"]
lfs = [lf_chf_positive, lf_chf_negation, lf_chf_tabular_format]
soft_labels_dict = {}

for comp in components:
    applier = PandasLFApplier(lfs=lfs)
    L = applier.apply(filtered_df)
print(LFAnalysis(L, lfs).lf_summary())